In [0]:
# Imports and configuration
import json
import time
import requests
from pyspark.sql import functions as F
import mlflow

# Source table — output from previous notebook
LLM_EXTRACTED   = "main.silver_techmart.llm_extracted"
# Target tables
SILVER_TAXONOMY = "main.silver_techmart.taxonomy"
SILVER_FLAGGED  = "main.silver_techmart.flagged_records"


In [0]:
# Groq configuration
GROQ_MODEL   = "llama-3.1-8b-instant"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
GROQ_API_KEY = dbutils.secrets.get(scope="techmart", key="groq-api-key")


In [0]:
# Load judge prompt from versioned config
PROMPTS_PATH = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/config/prompts.json"

with open(PROMPTS_PATH, "r") as f:
    prompts_config = json.load(f)

JUDGE_PROMPT_VERSION  = prompts_config["judge_prompt"]["version"]
JUDGE_TEMPLATE        = prompts_config["judge_prompt"]["template"]
APPROVED_TAXONOMY     = prompts_config["judge_prompt"]["approved_taxonomy"]

print(f"✅ Config loaded")
print(f"Judge prompt version: {JUDGE_PROMPT_VERSION}")
print(f"Approved taxonomy   : {APPROVED_TAXONOMY}")

In [0]:
# Read LLM extracted table
# We only judge records that were successfully extracted
df_extracted = spark.table(LLM_EXTRACTED).filter(
    F.col("llm_status") == "success"
)

print(f"Records to judge: {df_extracted.count()}")
display(df_extracted.select("product_id", "extracted_name", "extracted_brand", "extracted_sub_category"))

In [0]:
# Define the Judge LLM call function

def call_judge(name: str, brand: str, sub_category: str, api_key: str) -> dict:
    """
    Calls the Groq API to validate extracted product information.
    Returns a dict with approved (bool) and reason (str).
    Returns None if the call fails.
    """
    # Build the judge prompt from the versioned template
    prompt = JUDGE_TEMPLATE.format(
        approved_taxonomy=", ".join(APPROVED_TAXONOMY),
        name=name,
        brand=brand,
        sub_category=sub_category
    )

    payload = {
        "model": GROQ_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.1,
        "max_tokens": 100
    }

    response = requests.post(
        GROQ_API_URL,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}"
        },
        json=payload,
        timeout=30
    )

    if response.status_code != 200:
        return None

    response_text = response.json()["choices"][0]["message"]["content"]
    response_text = response_text.strip().replace("```json", "").replace("```", "").strip()

    return json.loads(response_text)

In [0]:
# CELL 4 — Run the Judge on all extracted records with MLflow tracing

mlflow.set_experiment("/Users/ts.maira@gmail.com/techmart-llm-extraction")

records = df_extracted.collect()
all_records = []

with mlflow.start_run(run_name=f"judge_{JUDGE_PROMPT_VERSION}"):

    mlflow.log_param("prompt_version", JUDGE_PROMPT_VERSION)
    mlflow.log_param("model", GROQ_MODEL)
    mlflow.log_param("provider", "groq")
    mlflow.log_param("total_records", len(records))

    for row in records:
        product_id   = row["product_id"]
        name         = row["extracted_name"]
        brand        = row["extracted_brand"]
        sub_category = row["extracted_sub_category"]

        start_time = time.time()

        try:
            result  = call_judge(name, brand, sub_category, GROQ_API_KEY)
            latency = round(time.time() - start_time, 3)

            if not result or "approved" not in result:
                raise ValueError("Missing keys in judge response")

            all_records.append({
                "product_id"            : str(product_id),
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_approved"        : str(result["approved"]),
                "judge_reason"          : result.get("reason", ""),
                "judge_latency_seconds" : str(latency),
                "prompt_version"        : JUDGE_PROMPT_VERSION
            })

        except Exception as e:
            latency = round(time.time() - start_time, 3)
            all_records.append({
                "product_id"            : str(product_id),
                "extracted_name"        : name,
                "extracted_brand"       : brand,
                "extracted_sub_category": sub_category,
                "judge_approved"        : "False",
                "judge_reason"          : f"Judge call failed: {str(e)}",
                "judge_latency_seconds" : str(latency),
                "prompt_version"        : JUDGE_PROMPT_VERSION
            })

        time.sleep(0.3)

    # Count approved and flagged from the single list
    approved_count = sum(1 for r in all_records if r["judge_approved"] == "True")
    flagged_count  = len(all_records) - approved_count

    mlflow.log_metric("approved_count", approved_count)
    mlflow.log_metric("flagged_count",  flagged_count)
    mlflow.log_metric("approval_rate",  round(approved_count / len(records), 3))

    print(f"✅ Judging complete")
    print(f"Approved: {approved_count}")
    print(f"Flagged : {flagged_count}")

In [0]:
# Write all records to a single table with judge status
# approved and flagged records are separated by the judge_approved column
# This makes querying simpler: filter by judge_approved = True/False

df_taxonomy = spark.createDataFrame(all_records)

(df_taxonomy.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable(SILVER_TAXONOMY))

print(f"✅ Written: {SILVER_TAXONOMY} ({len(all_records)} records)")
print(f"Approved: {len(approved_records)}")
print(f"Flagged : {len(flagged_records)}")

In [0]:
spark.table(SILVER_TAXONOMY).display()